In [21]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

llm = init_chat_model("openai:gpt-4o-mini")

In [22]:
class State(MessagesState):
    pass

graph_builder = StateGraph(State)

In [23]:
@tool
def get_weather(city: str):
    """Gets weather in City"""
    return f"The weather in {city} is sunny"

llm_with_tools = llm.bind_tools([get_weather])

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [24]:
tool_node = ToolNode(
    tools=[get_weather]
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

In [ ]:
graph.invoke(  
    {
        "messages": [
            {
                "role": "user",
                "content": "what is the weather in korea"
            }
        ]
    }
)

_IncompleteInputError: incomplete input (3168932218.py, line 8)